In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
path = "/content/drive/MyDrive/Zindi CO2 Challenge/"

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load datasets
train = pd.read_csv(path+'train.csv')
test = pd.read_csv(path+'test.csv')

# Save IDs for the final submission
test_ids = test['ID_LAT_LON_YEAR_WEEK']

In [7]:
# Drop columns with more than 90% missing values
n_samples = len(train)
cols_to_drop = [col for col in train.columns if train[col].isnull().sum() > 0.9 * n_samples]
train = train.drop(columns=cols_to_drop + ['ID_LAT_LON_YEAR_WEEK'])
test = test.drop(columns=cols_to_drop + ['ID_LAT_LON_YEAR_WEEK'])

# Fill remaining missing values with the mean
train = train.fillna(train.mean())
test = test.fillna(test.mean())

# Define features (X) and target (y)
X = train.drop('emission', axis=1)
y = train['emission']

In [8]:
# Split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train Random Forest Model
rf = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=1, random_state=42)
rf.fit(X_train, y_train)

# Check Validation Error (RootMeanSquareError)
val_preds = rf.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
print(f"Local Validation RMSE: {rmse:.4f}")

Local Validation RMSE: 29.8161


In [10]:
# Predict on test data
test_preds = rf.predict(test)

# Create submission dataframe
submission = pd.DataFrame({
    'ID_LAT_LON_YEAR_WEEK': test_ids,
    'emission': test_preds
})

# Save to CSV
submission.to_csv(path+'co2_submission.csv', index=False)